# 🩺 MedGemma DocAI — Kaggle Hosting
Run all cells top to bottom. The public ngrok URL will appear in the last cell.

In [1]:
# --- STEP 1: Install all required libraries ---
!pip install --quiet torch torchvision torchaudio transformers accelerate gradio pyngrok -q

In [2]:
# --- STEP 2: Login to HuggingFace using Kaggle Secrets ---
# HOW TO ADD YOUR TOKEN:
# 1. In this notebook, click the lock icon (🔒 'Add-ons' > 'Secrets') in the left sidebar
# 2. Add a new secret: Name = HF_TOKEN, Value = your HuggingFace token
# 3. Toggle the switch to enable it for this notebook
# Get your token from: https://huggingface.co/settings/tokens

from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

secrets = UserSecretsClient()
hf_token = secrets.get_secret("HF_TOKEN")
login(token=hf_token)
print("✅ Logged in to HuggingFace!")

✅ Logged in to HuggingFace!


In [3]:
# --- STEP 3: Load MedGemma model ---
from transformers import pipeline
import torch

MODEL_ID = "google/medgemma-4b-it"

pipe = pipeline(
    "image-text-to-text",
    model=MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

print("✅ MedGemma model loaded successfully!")

config.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

✅ MedGemma model loaded successfully!


In [4]:
# --- STEP 4: Define inference function (identical to your Colab version) ---
from PIL import Image
import traceback

def medgemma_infer(text_prompt: str):
    """
    Text-only handler for MedGemma.
    - Accepts a plain text input.
    - Returns safe and readable responses.
    - Handles empty input and runtime errors gracefully.
    """
    try:
        if not text_prompt or text_prompt.strip() == "":
            return "⚠️ Please enter a valid text prompt."

        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": text_prompt}
                ]
            }
        ]

        output_text = pipe(
            text=messages,
            max_new_tokens=1024,
            do_sample=False
        )
        return output_text[0]['generated_text'][1]['content']

    except torch.cuda.OutOfMemoryError:
        return "❌ GPU memory exhausted. Please restart your runtime or use a shorter prompt."

    except Exception as e:
        error_msg = traceback.format_exc(limit=1)
        return f"❌ Error during inference: {str(e)}\n\nDetails:\n{error_msg}"

print("✅ Inference function ready!")

✅ Inference function ready!


In [5]:
# --- STEP 5: Build Gradio interface ---
import gradio as gr

iface = gr.Interface(
    fn=medgemma_infer,
    inputs=[
        gr.Textbox(
            label="💬 Enter your prompt (e.g., 'Describe this X-ray')",
            lines=2,
            placeholder="Type here..."
        ),
    ],
    outputs=gr.Markdown(label="Output"),
    title="🩺 MedGemma - Medical AI Assistant",
    theme="gradio/soft",
)

print("✅ Gradio interface built!")

theme_schema%400.0.3.json: 0.00B [00:00, ?B/s]

✅ Gradio interface built!


In [6]:
# --- STEP 6: Set up ngrok tunnel ---
# HOW TO ADD YOUR NGROK TOKEN:
# 1. Same Kaggle Secrets panel (🔒 sidebar)
# 2. Add: Name = NGROK_TOKEN, Value = your ngrok authtoken
# Get your token from: https://dashboard.ngrok.com/get-started/your-authtoken

from pyngrok import ngrok, conf

ngrok_token = secrets.get_secret("NGROK_TOKEN")
ngrok.kill()  # kill any existing tunnels
conf.get_default().auth_token = ngrok_token

public_url = ngrok.connect(7860)
print(f"🌍 Your public API URL: {public_url.public_url}")
print(f"📡 Use this in your local project's gradio_client")

🌍 Your public API URL: https://malapportioned-synostotic-freeda.ngrok-free.dev                     
📡 Use this in your local project's gradio_client


In [7]:
# --- STEP 7: Launch! ---
iface.launch(server_name="0.0.0.0", server_port=7860, quiet=True)

* Running on public URL: https://54c0700df362bcfb42.gradio.live
